---
title: "State and Graph Control"
draft: true
categories: [agents, workflows, langgraph]
---

LangGraph starts with state, nodes, and transitions. The useful design work happens before the model call: decide which facts belong in state, which updates are append-only, and which transitions are legal.

## State is a contract

Define a state schema for the evidence brief with fields such as `question`, `scope`, `plan`, `sources`, `claims`, `contradictions`, `draft`, `review`, `revision_count`, and `status`. Separate durable artifacts from transient prompts. Every field needs an owner and an update rule.

Use `TypedDict` or another serializable schema for graph state. Use a separate validation model at external boundaries. Avoid placing open file handles, clients, iterators, or opaque model objects in checkpointed state.

## The graph API

Build the smallest graph with `StateGraph`, `START`, and `END`. Add nodes for intake, planning, drafting, and export. Add ordinary edges for mandatory transitions and conditional edges for decisions such as “more evidence needed?” or “review approved?”. Compile before invoking.

Then introduce reducers for fields that receive parallel updates. A reducer defines how independent node outputs combine. Without one, a fan-in stage can silently overwrite evidence collected by another branch.

## Commands and transition ownership

Use `Command` when a node must both update state and select the next node. Keep routing decisions close to the stage that owns the relevant invariant. A router should not mutate the draft, and a drafting node should not secretly decide whether review is required.

## Deliverable and experiment

Implement a graph with a fake model and fixture documents. Add tests for every legal route, missing required state, empty evidence, and an attempt to skip review. Draw the state transition diagram from the implementation and compare it with the one-page contract from Chapter 01.

The main experiment removes one state field at a time and observes whether the workflow still appears to succeed. These are hidden-state failures: the final prose may look plausible while the graph can no longer explain why it made a decision.
